# DIAMOND: Diffusion as a World Model — From Scratch

**Paper:** [Diffusion for World Modeling: Visual Details Matter in Atari](https://arxiv.org/abs/2405.12399) (NeurIPS 2024 Spotlight)

---

### The Big Idea

IRIS compressed images into discrete tokens before modelling them with a GPT. That compression *blurs* the 1-2 pixel sprites that carry reward in Atari games like *Asterix* and *Breakout*. **DIAMOND** throws away the tokenizer entirely and models the next frame as a **continuous pixel-space diffusion** process.

- **Input:** past 4 frames (channel-concatenated) + past 4 actions + noise level σ
- **Network:** small 2D Conv U-Net (~4.4 M params) with Adaptive GroupNorm for action conditioning
- **Loss:** EDM preconditioning (Karras 2022) — the network always predicts the clean image
- **Sampling:** only **3 Euler steps** per frame (DDPM would need 16+)
- **Training:** denoiser + reward/termination head trained from a replay buffer; policy trained *entirely in dreams* with REINFORCE + λ-returns

### Architecture

```
past 4 frames (12 ch) ─┐
noisy target (3 ch)   ─┼─▶ Conv U-Net ─▶ clean next frame
σ → sinusoidal embed  ─┤                 (via EDM preconditioning)
past 4 actions → MLP ──┘   (AdaGN in every res block)
```

### Environment: Atari Breakout (downscaled, grayscale-RGB)

We use a Breakout env at 64×64, 4 channels of grayscale broadcast to 3 RGB to keep the U-Net simple. A full DIAMOND run needs ~2.9 days on an RTX 4090; this notebook is a **compressed proof-of-concept** that runs in ~30 min on a Colab T4 — the denoiser learns to dream plausible frames, and you can compare 1-step vs 3-step sampling exactly like Figure 4 of the paper.

**Runtime:** ~25–35 min on T4 GPU

In [ ]:
!pip install -q gymnasium[atari,accept-rom-license] ale-py matplotlib tqdm

import os, math, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('medium')

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## 1. The Breakout Environment (64×64 grayscale-RGB)

We wrap `ALE/Breakout-v5` so it returns 64×64 grayscale frames broadcast to 3 channels, and we apply standard Atari preprocessing:
- Frame skip = 4
- Sticky actions (25% stick probability)
- Max over last 2 raw frames (to avoid flickering sprites)
- `FIRE` on reset (so the ball actually launches)

We also shrink the action space to the 4 meaningful actions: `NOOP, FIRE, RIGHT, LEFT`.

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import AtariPreprocessing, TransformObservation

IMG_SIZE = 64       # DIAMOND paper uses 64x64
N_ACTIONS = 4       # NOOP, FIRE, RIGHT, LEFT

def make_env():
    env = gym.make('ALE/Breakout-v5', frameskip=1, repeat_action_probability=0.25,
                   full_action_space=False, render_mode='rgb_array')
    env = AtariPreprocessing(env, frame_skip=4, grayscale_obs=True, screen_size=IMG_SIZE,
                             terminal_on_life_loss=True, noop_max=30, scale_obs=False)
    return env

env = make_env()
obs, info = env.reset(seed=SEED)
print(f'Action space: {env.action_space}')
print(f'Observation shape: {obs.shape}, dtype={obs.dtype}')

# Visualize a few random-policy frames
frames = [obs]
for _ in range(7):
    obs, r, term, trunc, _ = env.step(env.action_space.sample())
    frames.append(obs)
    if term or trunc: obs, _ = env.reset()

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for ax, f in zip(axes, frames):
    ax.imshow(f, cmap='gray', vmin=0, vmax=255); ax.axis('off')
fig.suptitle('Breakout — grayscale, 64×64, 4-frame skip', fontsize=11)
plt.tight_layout(); plt.show()

---
## 2. Collect a Replay Buffer with a Random Policy

DIAMOND trains from a replay buffer. In the paper this buffer grows interleaved with training (100 real steps per epoch). For this notebook we start with a fixed buffer of ~8000 transitions collected from a random policy — enough to teach the denoiser the visual dynamics of bricks, paddle, and ball.

We store episodes as sequences so the dataloader can cleanly sample `(obs_{t-3..t}, a_{t-3..t}, obs_{t+1}, r_{t+1}, done_{t+1})` windows.

In [ ]:
class ReplayBuffer:
    """Stores episodes as 64x64 uint8 grayscale frames + actions + rewards + dones."""
    def __init__(self):
        self.episodes = []  # list of dicts

    def add(self, obs, act, rew, done):
        self.episodes.append({
            'obs': np.stack(obs).astype(np.uint8),      # (T+1, 64, 64)
            'act': np.array(act, dtype=np.int64),        # (T,)
            'rew': np.array(rew, dtype=np.float32),      # (T,)
            'done': np.array(done, dtype=np.float32),    # (T,)
        })

    def total_steps(self):
        return sum(len(e['act']) for e in self.episodes)

    def sample_window(self, L=4):
        """Sample (obs[t-L+1..t+1], a[t-L+1..t], r[t+1], done[t+1]).
        Returns:
          past_obs: (L, 64, 64)   - the L frames ending at time t
          actions:  (L,)          - the L actions ending at a_t
          next_obs: (64, 64)      - the ground-truth frame at time t+1
          reward:   float         - r_{t+1}
          done:     float         - done_{t+1}
        """
        while True:
            ep = self.episodes[np.random.randint(len(self.episodes))]
            T = len(ep['act'])
            if T >= L + 1:
                break
        t = np.random.randint(L - 1, T)  # a[t] exists, obs[t+1] exists
        past_obs = ep['obs'][t - L + 1: t + 1]    # (L, 64, 64)
        actions  = ep['act'][t - L + 1: t + 1]    # (L,)
        next_obs = ep['obs'][t + 1]
        reward = ep['rew'][t]
        done = ep['done'][t]
        return past_obs, actions, next_obs, reward, done

    def sample_batch(self, batch_size, L=4):
        po, ac, no, rw, dn = [], [], [], [], []
        for _ in range(batch_size):
            a, b, c, d, e = self.sample_window(L)
            po.append(a); ac.append(b); no.append(c); rw.append(d); dn.append(e)
        past_obs = torch.from_numpy(np.stack(po)).float() / 255.0     # (B, L, 64, 64)
        actions  = torch.from_numpy(np.stack(ac)).long()              # (B, L)
        next_obs = torch.from_numpy(np.stack(no)).float() / 255.0     # (B, 64, 64)
        rew = torch.from_numpy(np.stack(rw)).float()
        dn  = torch.from_numpy(np.stack(dn)).float()
        return past_obs, actions, next_obs, rew, dn


# Collect
buffer = ReplayBuffer()
env = make_env()
TARGET_STEPS = 8000

pbar = tqdm(total=TARGET_STEPS, desc='Collecting random episodes')
while buffer.total_steps() < TARGET_STEPS:
    obs, _ = env.reset()
    obs_list, act_list, rew_list, done_list = [obs], [], [], []
    term = trunc = False
    while not (term or trunc):
        a = env.action_space.sample()
        obs, r, term, trunc, _ = env.step(a)
        obs_list.append(obs); act_list.append(a); rew_list.append(r)
        done_list.append(float(term or trunc))
    buffer.add(obs_list, act_list, rew_list, done_list)
    pbar.update(len(act_list))
pbar.close()
print(f'\nCollected {buffer.total_steps()} steps across {len(buffer.episodes)} episodes.')
print(f'Avg episode length: {buffer.total_steps()/len(buffer.episodes):.1f}')
print(f'Total reward collected: {sum(e["rew"].sum() for e in buffer.episodes):.1f}')

---
## 3. The Denoiser: a Small Conv U-Net

The heart of DIAMOND. The network `F_θ` takes:
- the last `L = 4` past frames, stacked along the channel axis (4 channels, grayscale)
- the noisy target frame `x_σ` (1 channel)
- so `5` input channels total (we keep it grayscale → simpler than the paper's RGB × 4)
- a noise-level embedding and action embedding, both injected via **Adaptive GroupNorm** (AdaGN) in every residual block

Architecture: 3-level downsample U-Net, 64 base channels, 2 res blocks per level, ~2 M params. Smaller than the paper (4.4 M) to fit Colab-T4 runtime.

In [ ]:
def sinusoidal_embed(t, dim=256):
    """Standard sinusoidal embedding for scalar t (shape (B,)) → (B, dim)."""
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    args = t[:, None].float() * freqs[None]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class AdaGNResBlock(nn.Module):
    """GroupNorm → SiLU → Conv twice, with scale/shift from a conditioning vector."""
    def __init__(self, in_c, out_c, cond_dim, groups=8):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_c)
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1)
        self.norm2 = nn.GroupNorm(groups, out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1)
        self.cond = nn.Linear(cond_dim, 2 * out_c)
        self.skip = nn.Conv2d(in_c, out_c, 1) if in_c != out_c else nn.Identity()

    def forward(self, x, cond):
        h = self.conv1(F.silu(self.norm1(x)))
        # Adaptive scale/shift from the condition vector
        scale, shift = self.cond(cond).chunk(2, dim=-1)
        h = self.norm2(h) * (1 + scale[:, :, None, None]) + shift[:, :, None, None]
        h = self.conv2(F.silu(h))
        return h + self.skip(x)


class DiamondUNet(nn.Module):
    """A compact DIAMOND-style denoiser.

    Input:
      x_noisy : (B, 1, 64, 64) — noisy target frame
      past    : (B, L, 64, 64) — L past frames concatenated on channel
      actions : (B, L)         — past L action ids
      sigma   : (B,)            — noise level (scalar per sample)
    Output:
      F_theta output, to be combined with EDM preconditioning externally.
    """
    def __init__(self, L=4, n_actions=N_ACTIONS, base=64, cond_dim=256, groups=8):
        super().__init__()
        self.L = L
        self.cond_dim = cond_dim

        # Conditioning: sigma embedding + action embedding, summed
        self.sigma_mlp = nn.Sequential(
            nn.Linear(cond_dim, cond_dim), nn.SiLU(), nn.Linear(cond_dim, cond_dim))
        self.act_embed = nn.Embedding(n_actions, cond_dim // L)
        self.act_mlp = nn.Sequential(
            nn.Linear(cond_dim, cond_dim), nn.SiLU(), nn.Linear(cond_dim, cond_dim))

        # Encoder: 64 → 32 → 16 → 8
        in_c = 1 + L   # noisy target (1) + L past frames
        self.in_conv = nn.Conv2d(in_c, base, 3, padding=1)
        self.down1 = AdaGNResBlock(base, base, cond_dim, groups)
        self.down2 = AdaGNResBlock(base, base * 2, cond_dim, groups)
        self.pool = nn.AvgPool2d(2)
        self.down3 = AdaGNResBlock(base * 2, base * 2, cond_dim, groups)
        self.down4 = AdaGNResBlock(base * 2, base * 4, cond_dim, groups)

        self.mid1 = AdaGNResBlock(base * 4, base * 4, cond_dim, groups)
        self.mid2 = AdaGNResBlock(base * 4, base * 4, cond_dim, groups)

        # Decoder
        self.up1 = AdaGNResBlock(base * 4 + base * 2, base * 2, cond_dim, groups)
        self.up2 = AdaGNResBlock(base * 2, base * 2, cond_dim, groups)
        self.upsample = nn.Upsample(scale_factor=2, mode='nearest')
        self.up3 = AdaGNResBlock(base * 2 + base, base, cond_dim, groups)
        self.up4 = AdaGNResBlock(base, base, cond_dim, groups)

        self.out_norm = nn.GroupNorm(groups, base)
        self.out_conv = nn.Conv2d(base, 1, 3, padding=1)

    def _build_cond(self, sigma, actions):
        # sigma embedding: EDM uses c_noise = 0.25 * ln(sigma)
        c_noise = 0.25 * torch.log(sigma.clamp_min(1e-8))
        s = sinusoidal_embed(c_noise, self.cond_dim)
        s = self.sigma_mlp(s)
        # actions: embed each, concat along dim, then project
        a = self.act_embed(actions)                 # (B, L, cond/L)
        a = a.reshape(actions.shape[0], -1)          # (B, cond)
        a = self.act_mlp(a)
        return s + a                                 # (B, cond)

    def forward(self, x_noisy, past, sigma, actions):
        cond = self._build_cond(sigma, actions)
        x = torch.cat([x_noisy, past], dim=1)        # (B, 1+L, 64, 64)
        h0 = self.in_conv(x)

        h1 = self.down1(h0, cond)          # 64
        h1 = self.down2(h1, cond)          # 64, base*2
        p1 = self.pool(h1)                  # 32
        h2 = self.down3(p1, cond)          # 32
        h2 = self.down4(h2, cond)          # 32, base*4
        p2 = self.pool(h2)                  # 16

        m = self.mid1(p2, cond)
        m = self.mid2(m, cond)

        u = self.upsample(m)               # 32
        u = torch.cat([u, h2], dim=1)
        u = self.up1(u, cond)
        u = self.up2(u, cond)

        u = self.upsample(u)               # 64
        u = torch.cat([u, h1], dim=1)
        u = self.up3(u, cond)
        u = self.up4(u, cond)

        out = self.out_conv(F.silu(self.out_norm(u)))
        return out                          # (B, 1, 64, 64)


denoiser = DiamondUNet().to(device)
n_params = sum(p.numel() for p in denoiser.parameters())
print(f'DIAMOND denoiser: {n_params/1e6:.2f}M parameters')
print('Input: 5 channels (noisy target + 4 past frames)')
print('Conditioning: sigma embedding + 4 past actions, injected via AdaGN')

---
## 4. EDM Preconditioning (Karras 2022)

Rather than making `F_θ` predict noise `ε` (the DDPM parameterization), EDM wraps the network so the **externally visible output is always an estimate of the clean image**, no matter the noise level:

$$D_\theta(x; \sigma) = c_{\text{skip}}(\sigma)\,x + c_{\text{out}}(\sigma)\,F_\theta\big(c_{\text{in}}(\sigma)\,x,\;c_{\text{noise}}(\sigma)\big)$$

with

$$c_{\text{skip}} = \frac{\sigma_{\text{data}}^2}{\sigma^2 + \sigma_{\text{data}}^2}, \quad c_{\text{out}} = \frac{\sigma\,\sigma_{\text{data}}}{\sqrt{\sigma^2 + \sigma_{\text{data}}^2}}, \quad c_{\text{in}} = \frac{1}{\sqrt{\sigma^2 + \sigma_{\text{data}}^2}}, \quad c_{\text{noise}} = \tfrac14 \ln \sigma$$

DIAMOND trains with `ln σ ~ 𝒩(-0.4, 1.2²)` and `σ_data = 0.5`. The loss is an L2 between the clean target and `D_θ(x_σ; σ)`, weighted by EDM's uncertainty weighting.

**Why this matters:** with DDPM parameterization, `F_θ` has to predict noise at high σ — but at high σ the noise is nearly the input itself, so the "clean image estimate" becomes useless. EDM's output is always the clean image, so even a single step of sampling gives a meaningful prediction.

In [ ]:
class EDMPrecond:
    """Wraps a raw network F_theta with EDM preconditioning and loss weighting."""
    def __init__(self, sigma_data=0.5):
        self.sigma_data = sigma_data

    def coefs(self, sigma):
        s2 = sigma ** 2
        sd2 = self.sigma_data ** 2
        c_skip = sd2 / (s2 + sd2)
        c_out  = sigma * self.sigma_data / torch.sqrt(s2 + sd2)
        c_in   = 1.0 / torch.sqrt(s2 + sd2)
        c_weight = (s2 + sd2) / (sigma * self.sigma_data).clamp_min(1e-8) ** 2
        return c_skip, c_out, c_in, c_weight

    def denoise(self, net, x_noisy, past, sigma, actions):
        """Apply EDM preconditioning: returns the clean-image estimate D_theta(x; sigma)."""
        c_skip, c_out, c_in, _ = self.coefs(sigma)
        c_skip = c_skip.view(-1, 1, 1, 1)
        c_out  = c_out.view(-1, 1, 1, 1)
        c_in   = c_in.view(-1, 1, 1, 1)
        F_out = net(c_in * x_noisy, past, sigma, actions)
        return c_skip * x_noisy + c_out * F_out

    def loss(self, net, clean, past, actions):
        """EDM training loss. clean: (B,1,64,64) in [-1,1] space."""
        B = clean.shape[0]
        # Sample ln sigma ~ N(-0.4, 1.2^2)
        ln_sigma = torch.randn(B, device=clean.device) * 1.2 - 0.4
        sigma = ln_sigma.exp()
        noise = torch.randn_like(clean) * sigma.view(-1, 1, 1, 1)
        x_noisy = clean + noise

        D_theta = self.denoise(net, x_noisy, past, sigma, actions)
        _, _, _, c_weight = self.coefs(sigma)
        c_weight = c_weight.view(-1, 1, 1, 1)
        # Clip weight to stabilize training
        c_weight = c_weight.clamp(max=10.0)
        mse = ((D_theta - clean) ** 2)
        return (c_weight * mse).mean()

    @torch.no_grad()
    def sample(self, net, past, actions, n_steps=3, sigma_max=5.0, sigma_min=0.02, rho=7.0):
        """Heun / Euler sampler (simple Euler here) with EDM sigma schedule. Returns clean frames in [-1,1]."""
        B, L, H, W = past.shape
        device = past.device

        # EDM sigma schedule (Karras)
        t = torch.linspace(0, 1, n_steps + 1, device=device)
        sigmas = (sigma_max ** (1 / rho) + t * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))) ** rho
        sigmas = torch.cat([sigmas, torch.zeros(1, device=device)])  # terminate at 0

        # Start from pure noise at sigma_max
        x = torch.randn(B, 1, H, W, device=device) * sigmas[0]
        for i in range(n_steps):
            sig = sigmas[i].expand(B)
            D = self.denoise(net, x, past, sig, actions)
            d = (x - D) / sigmas[i].clamp_min(1e-8)
            dt = sigmas[i + 1] - sigmas[i]
            x = x + d * dt
        return x.clamp(-1.0, 1.0)


edm = EDMPrecond(sigma_data=0.5)
print('EDM preconditioning ready. Training noise ln σ ~ N(-0.4, 1.2²), σ_data = 0.5.')

---
## 5. Train the Denoiser

We train with `[-1, 1]` normalized grayscale frames (recommended by EDM). Batch 64, lr 1e-4, AdamW. We run about 4 000 steps in this notebook — enough for the denoiser to learn bricks, paddle, and ball dynamics, though the paper runs orders of magnitude longer.

In [ ]:
def to_unit(x):
    """[0,1] → [-1,1]"""
    return x * 2.0 - 1.0

def from_unit(x):
    """[-1,1] → [0,1]"""
    return (x + 1.0) / 2.0

opt = torch.optim.AdamW(denoiser.parameters(), lr=1e-4, weight_decay=1e-4)
DENOISER_STEPS = 4000
BATCH = 64
L = 4

losses = []
denoiser.train()
pbar = tqdm(range(DENOISER_STEPS), desc='Training DIAMOND denoiser')
for step in pbar:
    past, actions, next_obs, rew, dn = buffer.sample_batch(BATCH, L)
    past = to_unit(past).to(device)                  # (B, L, 64, 64)
    next_obs = to_unit(next_obs).unsqueeze(1).to(device)  # (B, 1, 64, 64)
    actions = actions.to(device)

    loss = edm.loss(denoiser, next_obs, past, actions)

    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(denoiser.parameters(), 5.0)
    opt.step()

    losses.append(loss.item())
    if step % 200 == 0:
        pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.4f}')

print(f'\nFinal smoothed loss: {np.mean(losses[-100:]):.4f}')
plt.figure(figsize=(8, 3))
w = 50
sm = [np.mean(losses[max(0, i-w):i+1]) for i in range(len(losses))]
plt.plot(sm, color='#D97757')
plt.xlabel('Step'); plt.ylabel('EDM weighted MSE')
plt.title('DIAMOND denoiser training loss'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 6. 1 Step vs 3 Steps vs 10 Steps (The Mode Problem)

This is the experiment from Figure 4 of the paper. At 1 step, the network returns the conditional mean, which averages all possible futures and looks blurry. At 3 steps, the sampler collapses onto a single mode and the frame is crisp. Past 3 steps the gains are marginal.

In [ ]:
denoiser.eval()

# Sample one window and dream the next frame with different step counts
past, actions, next_obs, _, _ = buffer.sample_batch(8, L)
past_n = to_unit(past).to(device)
actions_d = actions.to(device)
gt = next_obs  # (B, 64, 64) in [0,1]

pred_1 = edm.sample(denoiser, past_n, actions_d, n_steps=1)
pred_3 = edm.sample(denoiser, past_n, actions_d, n_steps=3)
pred_10 = edm.sample(denoiser, past_n, actions_d, n_steps=10)

pred_1 = from_unit(pred_1).clamp(0, 1).squeeze(1).cpu()
pred_3 = from_unit(pred_3).clamp(0, 1).squeeze(1).cpu()
pred_10 = from_unit(pred_10).clamp(0, 1).squeeze(1).cpu()

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
row_titles = ['Ground truth', '1 step (blur)', '3 steps (crisp)', '10 steps']
for i in range(8):
    axes[0, i].imshow(gt[i], cmap='gray', vmin=0, vmax=1)
    axes[1, i].imshow(pred_1[i], cmap='gray', vmin=0, vmax=1)
    axes[2, i].imshow(pred_3[i], cmap='gray', vmin=0, vmax=1)
    axes[3, i].imshow(pred_10[i], cmap='gray', vmin=0, vmax=1)
    for r in range(4): axes[r, i].axis('off')

for r, t in enumerate(row_titles):
    axes[r, 0].set_ylabel(t, rotation=0, labelpad=50, fontsize=10,
                          color=('#7DA488' if r == 0 else ('#D97757' if r == 1 else '#5B8CB8')))

fig.suptitle('DIAMOND: 1 step averages modes (blur), 3 steps commits to one mode (sharp)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 7. Imagined Rollouts: Dreaming Breakout Frame by Frame

Now we roll the denoiser forward. Starting from 4 real frames, we ask the denoiser to predict frame 5 given a chosen action, then slide the window: frames 2-5 + new action → frame 6, etc. This is how the policy will eventually train in dreams.

In [ ]:
@torch.no_grad()
def dream_rollout(denoiser, edm, init_past, init_actions, action_seq, n_steps=3):
    """Roll the denoiser forward under a given action sequence.
    init_past: (L, 64, 64) in [0,1]
    init_actions: (L,) long
    action_seq: list[int] — future actions
    Returns list of frames in [0,1] including the L initial past frames.
    """
    denoiser.eval()
    L = init_past.shape[0]
    past = to_unit(init_past.clone()).unsqueeze(0).to(device)  # (1, L, 64, 64)
    acts = init_actions.clone().unsqueeze(0).to(device)        # (1, L)

    frames = [f.clone() for f in init_past]
    for a in action_seq:
        # Shift action window
        next_act = torch.tensor([[a]], device=device, dtype=torch.long)
        acts_win = torch.cat([acts[:, 1:], next_act], dim=1)

        pred = edm.sample(denoiser, past, acts_win, n_steps=n_steps)  # (1,1,64,64)
        pred_vis = from_unit(pred).clamp(0, 1).squeeze(0).squeeze(0).cpu()
        frames.append(pred_vis)

        # Slide window
        past = torch.cat([past[:, 1:], pred], dim=1)
        acts = acts_win
    return frames


# Grab 4 consecutive real frames as initial past
ep = buffer.episodes[np.random.randint(len(buffer.episodes))]
t0 = np.random.randint(L, len(ep['act']) - 15)
init_past = torch.from_numpy(ep['obs'][t0 - L + 1: t0 + 1]).float() / 255.0
init_acts = torch.from_numpy(ep['act'][t0 - L + 1: t0 + 1]).long()
# Future actions from the real episode (for side-by-side comparison)
future_actions = ep['act'][t0 + 1: t0 + 13].tolist()
real_future = ep['obs'][t0 + 1: t0 + 13]

dream_frames = dream_rollout(denoiser, edm, init_past, init_acts, future_actions, n_steps=3)

N = len(future_actions)
fig, axes = plt.subplots(2, N, figsize=(1.8 * N, 4))
for i in range(N):
    axes[0, i].imshow(real_future[i], cmap='gray', vmin=0, vmax=255)
    axes[0, i].set_title(f't+{i+1}', fontsize=8, color='#7DA488')
    axes[0, i].axis('off')
    axes[1, i].imshow(dream_frames[L + i], cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'dream', fontsize=8, color='#D97757')
    axes[1, i].axis('off')
fig.suptitle('Real Breakout vs DIAMOND dream (same start + same actions)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 8. Reward & Termination Head

In the paper, a **separate** small CNN-LSTM is trained on the same buffer to predict reward and termination from the past frames + next frame. We implement a tiny version: a 2-layer CNN over `(L+1)` frames → MLP → scalar reward + 2-way done logits.

In [ ]:
class RewardEndHead(nn.Module):
    def __init__(self, L=4, channels=32):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(L + 1, channels, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(channels, channels * 2, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(channels * 2, channels * 2, 4, 2, 1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.reward = nn.Linear(channels * 2, 1)
        self.done = nn.Linear(channels * 2, 2)

    def forward(self, past, next_frame):
        """past: (B, L, H, W), next_frame: (B, 1, H, W) — both in [-1,1]."""
        x = torch.cat([past, next_frame], dim=1)
        h = self.cnn(x)
        return self.reward(h).squeeze(-1), self.done(h)


rewnet = RewardEndHead(L=L).to(device)
rew_opt = torch.optim.Adam(rewnet.parameters(), lr=3e-4)

REW_STEPS = 1500
rewnet.train()
rew_losses = []
pbar = tqdm(range(REW_STEPS), desc='Training reward/end head')
for step in pbar:
    past, actions, next_obs, rew, dn = buffer.sample_batch(64, L)
    past_n = to_unit(past).to(device)
    next_n = to_unit(next_obs).unsqueeze(1).to(device)
    rew = rew.to(device); dn = dn.long().to(device)

    pred_r, pred_d = rewnet(past_n, next_n)
    loss_r = F.mse_loss(pred_r, rew)
    loss_d = F.cross_entropy(pred_d, dn, weight=torch.tensor([1.0, 20.0], device=device))
    loss = loss_r + loss_d

    rew_opt.zero_grad(); loss.backward(); rew_opt.step()
    rew_losses.append(loss.item())
    if step % 200 == 0:
        pbar.set_postfix(total=f'{np.mean(rew_losses[-50:]):.3f}',
                         r=f'{loss_r.item():.3f}', d=f'{loss_d.item():.3f}')
print('Reward/end head trained.')

---
## 9. Dream RL — Policy Trained Entirely in Imagination

Finally we train a small policy + value network with **REINFORCE + λ-returns** inside the dreamed rollouts, just like the paper (horizon 15, γ = 0.985, λ = 0.95). The policy never touches the real Breakout environment during updates — every frame, reward, and termination signal comes from the neural world model.

Because we only trained the denoiser for 4 000 steps and only on random-policy data, the dreams are not paper-quality; you should expect a modest improvement over random but not a superhuman Breakout agent. This mirrors the instructive progression of IRIS → DIAMOND: here you see the *mechanics* of dream RL with diffusion, and you can scale the training up exactly as described in the paper.

In [ ]:
class Policy(nn.Module):
    def __init__(self, L=4, n_actions=N_ACTIONS, channels=32):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(L, channels, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(channels, channels * 2, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(channels * 2, channels * 2, 4, 2, 1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.actor = nn.Linear(channels * 2, n_actions)
        self.critic = nn.Linear(channels * 2, 1)

    def forward(self, past):
        h = self.cnn(past)
        return self.actor(h), self.critic(h).squeeze(-1)


def lambda_returns(rewards, values, dones, gamma=0.985, lam=0.95):
    T = len(rewards)
    adv = [torch.zeros_like(rewards[0])] * T
    last = torch.zeros_like(rewards[0])
    for t in reversed(range(T)):
        next_v = values[t + 1] if t + 1 < T else torch.zeros_like(values[0])
        mask = 1.0 - dones[t]
        delta = rewards[t] + gamma * next_v * mask - values[t]
        last = delta + gamma * lam * mask * last
        adv[t] = last
    advantages = torch.stack(adv)
    returns = advantages + torch.stack(values)
    return advantages, returns


policy = Policy(L=L).to(device)
p_opt = torch.optim.Adam(policy.parameters(), lr=3e-4)

HORIZON = 15
DREAM_EPOCHS = 80
DREAM_BATCH = 16

dream_rewards_log = []
policy.train(); denoiser.eval(); rewnet.eval()

pbar = tqdm(range(DREAM_EPOCHS), desc='Dream RL')
for epoch in pbar:
    # Sample initial pasts from the buffer
    past_list, act_list = [], []
    for _ in range(DREAM_BATCH):
        p, a, _, _, _ = buffer.sample_window(L)
        past_list.append(p); act_list.append(a)
    past = to_unit(torch.from_numpy(np.stack(past_list)).float() / 255.0).to(device)
    acts = torch.from_numpy(np.stack(act_list)).long().to(device)

    log_probs, values, rewards_per_step, dones_per_step, entropies = [], [], [], [], []

    for step in range(HORIZON):
        logits, value = policy(past)
        dist = torch.distributions.Categorical(logits=logits)
        a = dist.sample()
        log_probs.append(dist.log_prob(a))
        values.append(value)
        entropies.append(dist.entropy())

        # Slide action window and dream next frame (3 EDM steps)
        new_acts = torch.cat([acts[:, 1:], a.unsqueeze(1)], dim=1)
        with torch.no_grad():
            next_frame = edm.sample(denoiser, past, new_acts, n_steps=3)
            pred_r, pred_d = rewnet(past, next_frame)
        rewards_per_step.append(pred_r.detach())
        dones_per_step.append(pred_d.argmax(-1).float().detach())

        past = torch.cat([past[:, 1:], next_frame], dim=1)
        acts = new_acts

    adv, ret = lambda_returns(rewards_per_step, values, dones_per_step)
    log_probs_t = torch.stack(log_probs)
    values_t = torch.stack(values)
    entropy = torch.stack(entropies).mean()

    # Normalize advantages per-batch for stable REINFORCE
    adv_norm = (adv - adv.mean()) / (adv.std() + 1e-6)
    loss_pi = -(log_probs_t * adv_norm.detach()).mean()
    loss_v  = F.mse_loss(values_t, ret.detach())
    loss = loss_pi + 0.5 * loss_v - 0.01 * entropy

    p_opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 5.0); p_opt.step()

    dream_rewards_log.append(torch.stack(rewards_per_step).sum(0).mean().item())
    if epoch % 10 == 0:
        pbar.set_postfix(dream_rew=f'{dream_rewards_log[-1]:+.2f}',
                         loss=f'{loss.item():.3f}')

plt.figure(figsize=(8, 3))
plt.plot(dream_rewards_log, color='#D97757')
plt.title('Total imagined return per episode (horizon 15)')
plt.xlabel('Dream epoch'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
## 10. Evaluate the Dream-Trained Policy in the Real Environment

Now we run the policy in the actual Breakout environment — it has **never** seen a real frame during training, only dreamed ones. We compare against a random baseline. In the paper, after full training, the policy reaches ~132 average score on Breakout (vs IRIS's 83). With our compressed training loop you can expect modest improvements over random — the goal of the notebook is to *see the mechanics work*, not to hit paper numbers.

In [ ]:
@torch.no_grad()
def evaluate(policy, n_episodes=10, greedy=False):
    env = make_env()
    policy.eval()
    scores = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        frame_buf = [obs] * L
        total = 0.0
        term = trunc = False
        while not (term or trunc):
            past = np.stack(frame_buf[-L:]).astype(np.float32) / 255.0
            past_n = torch.from_numpy(past * 2.0 - 1.0).unsqueeze(0).to(device)
            logits, _ = policy(past_n)
            if greedy:
                a = int(logits.argmax(dim=-1).item())
            else:
                a = int(torch.distributions.Categorical(logits=logits).sample().item())
            obs, r, term, trunc, _ = env.step(a)
            total += r
            frame_buf.append(obs)
        scores.append(total)
    return np.mean(scores), np.std(scores)


# Random baseline
def eval_random(n_episodes=10):
    env = make_env()
    scores = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        total = 0.0
        term = trunc = False
        while not (term or trunc):
            obs, r, term, trunc, _ = env.step(env.action_space.sample())
            total += r
        scores.append(total)
    return np.mean(scores), np.std(scores)

rand_mean, rand_std = eval_random(10)
pol_mean, pol_std = evaluate(policy, 10, greedy=False)

print('\n' + '=' * 50)
print('     DIAMOND DREAM-TRAINED POLICY — REAL BREAKOUT')
print('=' * 50)
print(f'  Random baseline : {rand_mean:.2f} ± {rand_std:.2f}')
print(f'  Dream policy    : {pol_mean:.2f} ± {pol_std:.2f}')
print(f'  Paper (full training): ~132')
print('=' * 50)

---
## Summary

We built **DIAMOND** from scratch — a world model that replaces IRIS's discrete-token autoregressive transformer with a continuous-pixel diffusion denoiser.

### The Three Components

| Component | What it does | Architecture | Analogy |
|---|---|---|---|
| **Denoiser** | Predict the next frame | Conv U-Net + AdaGN, EDM loss | Stable Diffusion as a world model |
| **Reward/End head** | Predict `r_{t+1}` and `done` | Small CNN | IRIS's scalar heads |
| **Actor-Critic** | Choose actions, estimate value | Small CNN policy | REINFORCE in dreams |

### Key Insights From the Experiment

1. **EDM preconditioning is the trick.** The loss we minimized is weighted MSE between the network's clean-image estimate and the ground-truth next frame. Because `D_θ(x; σ)` always returns a clean image, training is stable across the whole σ range and sampling works at 3 steps. Try swapping to a DDPM `ε`-prediction target and watch quality collapse.

2. **1 step = conditional mean = blur. 3 steps = mode selection = sharp.** You saw this directly in Section 6. This is why the paper uses 3 steps, not 1.

3. **Channel-concat + AdaGN beats sequence models at 64×64.** No attention, no tokenizer, no codebook — just a convnet that knows how to dream frames.

4. **Dream RL really works.** The policy updates only ever saw imagined frames. On a full paper-scale run the policy reaches ~132 on Breakout (vs IRIS's 83) while using ~20× fewer parameters.

### Scaling This to Paper Numbers

- Replace the fixed-buffer collection with DIAMOND's interleaved schedule: 100 real env steps → 400 denoiser grad steps → 400 policy grad steps, repeated for 500 epochs.
- Raise the denoiser channel width to 64 base / 4.4 M params, and train 50 × longer.
- Train the reward/end head as a CNN-LSTM, not an MLP.
- Use Heun's 2nd-order sampler instead of pure Euler (tiny quality win at 3 NFE).
- Follow the exact EDM schedule with `ρ = 7`, `σ_min = 0.002`, `σ_max = 20`.

### Going Further

- **DIAMOND on CS:GO.** Same recipe, 80× bigger — 330 M dynamics + 51 M upsampler, 12 days on a single RTX 4090, playable at 10 FPS.
- **LeWorld** and **DreamZero.** See Part 2 of this lecture for tiny end-to-end JEPAs and joint world-action models trained on internet video.
- **GameNGen / Genie 3.** Direct descendants: neural game engines built from diffusion world models.

### References
- [DIAMOND paper](https://arxiv.org/abs/2405.12399) | [Code](https://github.com/eloialonso/diamond)
- [EDM paper (Karras 2022)](https://arxiv.org/abs/2206.00364)
- [IRIS paper](https://arxiv.org/abs/2209.00588) (the baseline we beat)